# Notebook 01: Setup Dati & Validazione Pipeline
## Fase 1 del Piano Operativo (`results/final_experimental_report.md`)

**Obiettivi:**
1. **T1.1**: Verifica del dataset CIC-IoMT-2024 (13 classi target) e mapping label.
2. **T1.2**: Estrazione della finestra spazio-temporale $T=10$ pacchetti per i 6 campi di header (`PL, IAT, DIR, WIN, TTL, FLG`).
3. **T1.3**: Verifica della partizione stratificata deterministica a `seed=0` (70% Train+Validation / 30% Test; la Validation è il 10% del Train, quindi 63% / 7% / 30%).
4. **T1.4**: Istanziazione dei `DataLoader` PyTorch ($B=64$) e verifica padding `0.0` vs valori fisici scalati $[0.001, 1.0]$.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import pandas as pd

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import seed_everything
seed_everything(seed=0)
print(f"Repo Root: {REPO_ROOT}")
print("Determinismo globale impostato a seed=0.")

### 1.1 Localizzazione Dataset e Controllo Metadati (T1.1)
Cerchiamo il dataset CIC-IoMT-2024 nei percorsi standard (NAS del cluster `/nas/data/datasets/cic-iomt/` o directory locale `data/`).

In [ ]:
DATA_CANDIDATES = [
    Path('/nas/data/datasets/cic-iomt'),
    REPO_ROOT / 'dataset',
    REPO_ROOT / 'data',
    Path.home() / 'data' / 'cic-iomt'
]

DATA_DIR = None
for p in DATA_CANDIDATES:
    if p.exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    DATA_DIR = REPO_ROOT / 'data'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[WARN] Percorso dataset non trovato nei percorsi standard. Impostato a {DATA_DIR}")
else:
    print(f"[OK] Dataset individuato in: {DATA_DIR}")

# Check classes.txt and label_conv.json
CLASSES_FILE = DATA_DIR / 'classes.txt'
if CLASSES_FILE.exists():
    with open(CLASSES_FILE) as f:
        classes = [line.strip() for line in f if line.strip()]
    print(f"[OK] Trovate {len(classes)} classi in classes.txt:")
    for idx, c in enumerate(classes):
        print(f"  {idx:2d}: {c}")
else:
    print(f"[INFO] classes.txt verra generato al primo caricamento del dataset.")

### 1.2 Configurazione Input: $T=10$ Pacchetti, $F=6$ Campi Header (T1.2)
Configuriamo i parametri del dataset networking secondo il protocollo congelato in `results/final_experimental_report.md`.

In [ ]:
NUM_PKTS = 10
FIELDS = ['PL', 'IAT', 'DIR', 'WIN', 'TTL', 'FLG']
NUM_CLASSES = 13
BATCH_SIZE = 64
SEED = 0

print("Configurazione input spazio-temporale:")
print(f"- Orizzonte temporale T: {NUM_PKTS} pacchetti")
print(f"- Campi di intestazione F ({len(FIELDS)}): {FIELDS}")
print(f"- Task: Multiclasse ({NUM_CLASSES} classi)")
print(f"- Seed: {SEED}")

### 1.3 Creazione e Test dei DataLoader (T1.3 & T1.4)
Verifichiamo che i loader generino tensori di forma `[batch_size, 1, 10, 6]` con padding mascherato a `0.0`.

In [ ]:
from datasets.data_loader import get_loaders

trn_loader, val_loader, tst_loader, taskcla = get_loaders(
    datasets=['cic-iomt'],
    num_tasks=1,
    nc_first_task=NUM_CLASSES,
    nc_incr_tasks=0,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    seed=SEED,
    num_pkts=NUM_PKTS,
    fields=FIELDS
)
print("[SUCCESS] DataLoader istanziati con successo!")
print(f"  Train batches: {len(trn_loader[0])}")
print(f"  Val batches:   {len(val_loader[0])}")
print(f"  Test batches:  {len(tst_loader[0])}")

# Inspect the first batch
images, targets = next(iter(trn_loader[0]))
print(f"\nForma batch input:   {images.shape} (Atteso: [{BATCH_SIZE}, 1, {NUM_PKTS}, {len(FIELDS)}])")
print(f"Forma batch target:  {targets.shape}")
print(f"Valori input min/max: [{images.min():.4f}, {images.max():.4f}]")
print(f"Frazione padding 0.0: {(images == 0.0).float().mean() * 100:.2f}%")